# Video Game Sales: How Much Does Genre Explain?
**DBMS71008 Data Analytics | CA2 CRISP-DM Analytics Project**
**Author:** [YOUR NAME] | **Student number:** [STUDENT NUMBER]
**GitHub repository:** [REPOSITORY URL]

> *Guidance (delete all blockquote guidance before submitting): Each section below follows the CA2 brief. Explain what you are about to do, run the code, then interpret the output in a markdown cell straight after it. Charts or tables without interpretation do not count as evidence of understanding.*

**Contents**
1. Business Understanding
2. Data Understanding
3. Data Preparation
4. Modelling
5. Evaluation
6. Deployment and Recommendations
7. Reflection
8. Responsible AI Collaboration Statement

## 0. Setup
> *Imports, fixed random seed and data loading. Use relative or documented paths only. No absolute paths, passwords or API keys.*

In [28]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/Jack-O-Mahoney/DataAnalytics-CA2/main/Data/Raw/vgsales.csv"
df = pd.read_csv(url)
df.head(25)

,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82
3,4,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00
4,5,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37
5,6,Tetris,GB,1989.0,Puzzle,Nintendo,23.20,2.26,4.22,0.58,30.26
6,7,New Super Mario Bros.,DS,2006.0,Platform,Nintendo,11.38,9.23,6.50,2.90,30.01
7,8,Wii Play,Wii,2006.0,Misc,Nintendo,14.03,9.20,2.93,2.85,29.02
8,9,New Super Mario Bros. Wii,Wii,2009.0,Platform,Nintendo,14.59,7.06,4.70,2.26,28.62
9,10,Duck Hunt,NES,1984.0,Shooter,Nintendo,26.93,0.63,0.28,0.47,28.31


# 1. Business Understanding
> *Define the real problem before any algorithm. Say who needs the result, what decision it supports and what counts as success. (Brief 4.1)*

## 1.1 Context and Problem
There are thousands of game's getting released in the game industry but few earn large amounts of money.Publishers often choose which projects/games to fund based on the games genre yet sales can also depend on the platform, publisher and release timing. This Project asks how much genre explains about a game's global sales once those other factors are taken into account

## 1.2 Stakeholders and Users
Data Supplier: VGChartz, whoes data i got from Kaggle

Users: publishers deciding which projects to fund

Affected: Developers whose projects could be favoured or dropped because of genre assumptions.

## 1.3 Business / Practical Objective
Help a publisher decide how much genre effects projects when compareing proposals,relative to platform and timing.

## 1.4 Research Question and Analytics Objective
Question: How much of the difference in gloabal video game sales can be explained by genre and which genres corralate with higher sales when the platform,publisher and release year are taken into account?

Analytics objective: Fit linear regression and random forest regression models of global sales, each combared against each other and see which predicts better and what they show about genre.

## 1.5 Success Criteria

**Technical success criteria:**

Both models predict better than the baseline (the average sales), shown by a lower RMSE and MAE on the test data.

R² is reported for each model, including a genre-only version, to show how much of the variation in sales genre explains.

The residuals (prediction errors) are checked for obvious patterns.

Results stay similar when the data is split in a different way.

**Practical success criteria:**

The findings can be explained to a publisher in plain language.

The report says clearly whether genre is a strong or weak guide to sales and how certain that conclusion is.

## 1.6 Constraints and Assumptions

| Type | Area | Detail |
|---|---|---|
| Constraint | Data | Covers about 1980–2016, so it says nothing about the current market |
| Constraint | Data | Sales are VGChartz estimates, not official figures |
| Constraint | Data | Digital sales are not included |
| Constraint | Time | Three weeks to the 4 December deadline |
| Constraint | Resources | One person, using free tools only (Colab, GitHub, Python libraries) |
| Constraint | Privacy | Each row is a game, not a person, so there is little personal-data risk |
| Assumption | Interpretation | Findings show associations, not causes |
| Assumption | Data quality | The VGChartz estimates are accurate enough to compare genres |


## 1.7 Initial Project Plan
Look at the data and fix any problems in it (by 19 Nov). Start building the models by 20 Nov and finish them by 24 Nov. first a basline then linear regression and random forest regression, both predicting the log sales. Check how well they work (27 Nov), write the paper (1 Dec) and do final checks to make sure everything is correct (3 Dec).

# 2. Data Understanding
> *Show where the data came from, what one row represents, how variables were measured and whether the data can support the objective. (Brief 4.2)*

## 2.1 Initial Data Collection Report

Source:Kaggle Video Games Sales dataset origininaly made by VGChartz. Accessed 13/11/26.One File vgsales.csv. Covers 1980-2016. Sales are split into NA, EU, JP and other regions. Digital sales are not included.

## 2.2 Data Description Report
The dataset is a single table of video game sales. One row represents one game on one platform, so the same title can appear several times if it was released on different platforms.

In [15]:
# Size of the dataset and the type of each column
print(df.shape)
print(df.dtypes)

(16598, 11)
Rank              int64
Name                str
Platform            str
Year            float64
Genre               str
Publisher           str
NA_Sales        float64
EU_Sales        float64
JP_Sales        float64
Other_Sales     float64
Global_Sales    float64
dtype: object


In [18]:
# Number of different values in the category columns, and the range of years
print(df[["Platform", "Genre", "Publisher"]].nunique())
print(df["Year"].min(), df["Year"].max())

Platform      31
Genre         12
Publisher    578
dtype: int64
1980.0 2020.0


In [17]:
# Summary of the numeric columns
df.describe()

,Rank,Year,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
count,16598.000000,16327.000000,16598.000000,16598.000000,16598.000000,16598.000000,16598.000000
mean,8300.605254,2006.406443,0.264667,0.146652,0.077782,0.048063,0.537441
std,4791.853933,5.828981,0.816683,0.505351,0.309291,0.188588,1.555028
min,1.000000,1980.000000,0.000000,0.000000,0.000000,0.000000,0.010000
25%,4151.250000,2003.000000,0.000000,0.000000,0.000000,0.000000,0.060000
50%,8300.500000,2007.000000,0.080000,0.020000,0.000000,0.010000,0.170000
75%,12449.750000,2010.000000,0.240000,0.110000,0.040000,0.040000,0.470000
max,16600.000000,2020.000000,41.490000,29.020000,10.220000,10.570000,82.740000


**Interpretation:** 

The dataset has 16,598 rows and 11 columns. There are 12 genres, 31 platforms and 578 publishers. The Year is stored as a decimal number because it contains missing values. The Global Sales ranges from 0.01 to 82.74, but its median is much lower than its maximum, which suggests a few very large sellers. This is looked at further in the exploration section.

**Data dictionary**

| Variable | Meaning | Type | Units / valid range | Role |
|---|---|---|---|---|
| Rank | Position by global sales | Integer | 1 to about 16,600 | Drop: tooken from the target |
| Name | Game title | Text | n/a | Identifier |
| Platform | Console or system | Category (31) | n/a | Input |
| Year | Release year | Numeric | about 1980–2016 (a few 2017 and 2020 values are errors) | Input |
| Genre | Game genre | Category (12) | n/a | Input of interest |
| Publisher | Publishing company | Category (578) | n/a | Input |
| NA_Sales, EU_Sales, JP_Sales, Other_Sales | Sales by region | Numeric | millions of units, 0 or more | Drop: they add up to the target |
| Global_Sales | Total sales across regions | Numeric | millions of units, 0.01 to 82.74 | **Target** |

**Why some columns are dropped:**

 The regional sales columns add up to Global Sales, and the Rank is based on it. Using them as inputs would give the model the answer in advance causing data leakage, so they are excluded.

## 2.3 Data Exploration Report
> *Appropriate frequencies, proportions and descriptive statistics. Charts linked to analytical questions. Describe distribution shape, centre, spread, groups, relationships and unusual values. State observations cautiously, without implying causation.*

In [25]:
# Working copy for exploration (the raw data stays unchanged)
explore = df.copy()
explore["Log_Sales"] = np.log(explore["Global_Sales"])  # smallest sale is 0.01, so no zeros

NameError: name 'np' is not defined

**Interpretation:** *What does this show, why does it matter, what is uncertain and what action may be needed?*

## 2.4 Data Quality Report
> *Missing values by count, percentage and pattern. Duplicates, invalid values, inconsistent categories. Potential outliers and measurement artefacts. Representation, imbalance and likely bias. Prioritise the issues to address in Data Preparation.*

In [ ]:
# Missing values, duplicates, invalid years, rare categories, outliers

**Interpretation and prioritised issues:**
1.

# 3. Data Preparation
> *Only carry out preparation justified by the objective and the Data Understanding findings. Preserve the raw data and record the impact of each change. (Brief 4.3)*

## 3.1 Selection
> *Inclusion and exclusion rules, counts affected and possible impact on representation.*

## 3.2 Cleaning
> *Treatment of missing, invalid, inconsistent, duplicate and unusual values.*

## 3.3 Type and Format Corrections
> *Explicit numeric, date, category and unit conversions.*

## 3.4 Feature Construction, Encoding and Scaling
> *Formula, source variables and rationale. Why the representation suits the chosen technique. Keep fitted parameters where relevant.*

## 3.5 Preventing Data Leakage
> *Split into train and test data. Learn any imputation values, scalers, encoders and feature-selection rules from the training data only, and explain how the same fitted preparation is applied to the test data.*

## 3.6 Preparation Log
> *Add a row for every preparation decision as the project develops.*

| Issue | Action | Rationale / evidence | Assumption | Number affected | Risk / verification |
|---|---|---|---|---|---|
| | | | | | |

## 3.7 Validation of Prepared Data
> *Before-and-after profiles, re-run quality checks and compare distributions. Save the prepared data separately from the raw data.*

In [ ]:
# Before/after checks and save to Data/Processed/

# 4. Modelling
> *Choose a technique because it addresses the approved objective. (Brief 4.4)*

## 4.1 Task Type and Technique Choice
> *State the task type (regression), why it fits the objective, and alternatives considered.*

## 4.2 Features, Target and Train/Test Arrangement
> *Features and target. Train, validation and test arrangement. Fixed random seed.*

## 4.3 Baseline
> *A simple reference point, for example predicting the training-set mean.*

## 4.4 Candidate Models and Settings
> *Compare at least one meaningful alternative, setting or baseline. Justify important parameters instead of reporting defaults without comment.*

## 4.5 Interpretability, Complexity and Limitations
> *What the model can and cannot tell you.*

# 5. Evaluation
> *Connect the technical results to the original objectives. (Brief 4.5)*

## 5.1 Technical Results
> *Suitable measures, such as MAE, RMSE and R-squared, and residual plots. Compare with the baseline and the technical success criteria. Report the units.*

## 5.2 Practical Usefulness
> *Is the result useful for the stakeholders and the intended decision?*

## 5.3 Error Analysis, Overfitting and Generalisability
> *Sources of error and important failure cases. Signs of overfitting, underfitting, leakage or unstable findings.*

## 5.4 Assumptions, Limitations, Ethics and Bias
> *Assumptions, representativeness, privacy, fairness, transparency, potential misuse and human oversight.*

## 5.5 Decision
> *Proceed, revise the data or model, or reject the result. Give the reasons.*

# 6. Deployment and Recommendations
> *No live system is required. Explain how the findings could be delivered and governed in practice. (Brief 4.6)*

- Intended users and form of delivery:
- Actions supported, and actions that should not be automated:
- Data and model refresh requirements:
- Monitoring for performance, drift and errors:
- Access control, privacy and retention:
- Human oversight and feedback routes:
- Realistic future work:

# 7. Reflection
> *A concise personal reflection. (Brief 4.7)*

- What changed between the original plan and the final project?
- Which decision required the most judgement?
- What failed, and what was learned?
- What would be done differently with more time or better data?
- Which parts of the work can I explain confidently without assistance?

# 8. Responsible AI Collaboration Statement
> *300 to 500 words under this heading (AIAS Level 3). If no generative AI was used, state that explicitly. Address all seven points. (Brief 7.4)*

1. **Tools and versions used:**
2. **Tasks for which AI assistance was used:**
3. **Representative prompts (or link to a prompt log):**
4. **How code, factual claims, calculations and references were verified:**
5. **Outputs that were rejected, corrected or substantially modified:**
6. **What remained my own judgement and decision:**
7. **What I learned about the strengths and limitations of AI collaboration:**

# References
> *Credible sources used, in the referencing style required by the module. Read and check every source, including any suggested by AI tools.*